In [3]:
%load_ext autoreload
%autoreload 2
import warnings
import matplotlib.pyplot as plt
import numpy as np
import glob
import xarray as xr
import xbudget
import regionate
import xwmt
import xwmb
import xgcm
import cartopy.crs as ccrs
import CM4Xutils #needed to run pip install nc-time-axis
from regionate import MaskRegions, GriddedRegion

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Request HPC Resources

In [1]:
from dask_jobqueue import SLURMCluster  # setup dask cluster 
from dask.distributed import Client

log_directory="/vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/notebooks/logs"

cluster = SLURMCluster(
    cores=36,
    processes=1,
    memory='160GB',
    walltime='8:00:00',
    queue='compute',
    interface='ib0', 
log_directory = log_directory)
print(cluster.job_script())
cluster.scale(jobs=8)
cluster.wait_for_workers(4)
client = Client(cluster)
client

#!/usr/bin/env bash

#SBATCH -J dask-worker
#SBATCH -e /vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/notebooks/logs/dask-worker-%J.err
#SBATCH -o /vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/notebooks/logs/dask-worker-%J.out
#SBATCH -p compute
#SBATCH -n 1
#SBATCH --cpus-per-task=36
#SBATCH --mem=150G
#SBATCH -t 8:00:00

/vortexfs1/home/anthony.meza/miniforge3/envs/cm4x_chapter2/bin/python -m distributed.cli.dask_worker tcp://172.16.3.54:40812 --name dummy-name --nthreads 36 --memory-limit 149.01GiB --nanny --death-timeout 60 --interface ib0



<Client: 'tcp://172.16.3.54:40812' processes=7 threads=252, memory=1.02 TiB>

### Load in data

In [4]:
# Key water mass transformation budget terms
budget_terms = [
    # 'Eulerian_tendency', 'advection', 'diffusion', 
            'boundary_fluxes', #'convergent_mass_transport', 
           # 'mass_tendency', 'mass_source', 'spurious_numerical_mixing', 
           "surface_exchange_flux", "bottom_flux", "frazil_ice", 
            "surface_ocean_flux_advective_negative_rhs"]

other_budget_terms = ["surface_ocean_flux_advective_negative_rhs_heat", 
                     "surface_ocean_flux_advective_negative_rhs_salt", 
                     "surface_exchange_flux_heat", 
                     "surface_exchange_flux_salt", 
                     "frazil_ice_heat", 
                     "bottom_flux_heat",
                     "boundary_fluxes", 
                     # "mass_tendency", 
                     # "diffusion_heat", 
                     # "diffusion_salt",
                     # "spurious_numerical_mixing",
                     # "convergent_mass_transport"
                     ]
budget_terms= sorted(list(set(budget_terms) | set(other_budget_terms)))

In [5]:
decomp_budget_terms = ["surface_exchange_flux_advective_evaporation_salt", 
                    "surface_exchange_flux_advective_rain_and_ice_salt",
                    "surface_exchange_flux_advective_snow_salt",
                    "surface_exchange_flux_advective_rivers_salt", 
                    "surface_exchange_flux_advective_icebergs_salt",
                    "surface_exchange_flux_advective_virtual_precip_restoring_salt",
                    "surface_exchange_flux_advective_sea_ice_salt",
                    "surface_exchange_flux_nonadvective_basal_salt",
                    "surface_exchange_flux_nonadvective_longwave_heat", 
                    "surface_exchange_flux_nonadvective_shortwave_heat",
                    "surface_exchange_flux_nonadvective_sensible_heat",
                    "surface_exchange_flux_nonadvective_latent_heat", 
                    "surface_exchange_flux_advective_mass_transfer_heat"]

In [6]:
def decompose_boundary_fluxes(ds): 
    ds["boundary_fluxes_heat"] = ds["surface_ocean_flux_advective_negative_rhs_heat"] +\
                             ds["surface_exchange_flux_heat"] +\
                             ds["frazil_ice_heat"] + ds["bottom_flux_heat"]

    ds["boundary_fluxes_salt"] = ds["surface_ocean_flux_advective_negative_rhs_salt"] +\
                                     ds["surface_exchange_flux_salt"]

    ds["surface_boundary_fluxes_salt"] = 1 * ds["boundary_fluxes_salt"]
    ds["surface_boundary_fluxes_heat"] = ds["boundary_fluxes_heat"] - ds["bottom_flux_heat"]

In [31]:

datadir = lambda x="" : "/proj/ecco/CM4X/budget_sigma2_v1.2.0/" + x
datafiles = glob.glob(datadir("CM4Xp125*"))[20:]
datafiles = sorted(datafiles)

wmts = []

for (t, file) in enumerate(datafiles[0:2]): 
    print(file)
    ds = xr.open_mfdataset(
        file,
        data_vars="minimal",
        coords="minimal",
        compat="override",
        parallel=True,
        engine="zarr")
    ds = ds.fillna(0.)

    ds['mask'] = 1.0 * (
        (ds['geolat'] <= -50.5) * 
    (ds["deptho"].fillna(0.0) <= 1001.0))
    
    grid = CM4Xutils.ds_to_grid(ds)
    regions = MaskRegions(ds.mask, grid).region_dict
    antarctic = regions[0] #there are more in this list if there are multiple contours 
    region = GriddedRegion("antarctic", antarctic.lons_c, antarctic.lats_c, grid, 
                           ij=(antarctic.i_c, antarctic.j_c))
    selection_kwargs = {"sigma2_l_target":slice(36.25, 37.5), "yh":slice(None, 50)}
    ds_kwargs = selection_kwargs.copy()
    ds_kwargs["sigma2_l"] = ds_kwargs.pop("sigma2_l_target")

    with warnings.catch_warnings():
        warnings.simplefilter(action='ignore', category=FutureWarning)
    
        budgets_dict = xbudget.load_yaml("MOM6_AABW_updated.yaml")

        xbudget.collect_budgets(grid, budgets_dict)
        
        wmb = xwmb.WaterMassBudget(
            grid,
            budgets_dict, 
            region = region
        ) #if region not passed, the whole globe is taken
        wmb.mass_budget("sigma2", greater_than=True, default_bins=False, 
                        integrate=False, along_section=False)
        
        wmt = wmb.wmt[budget_terms].sel(selection_kwargs).rename({"sigma2_l_target":"sigma2_l"})
        wmt_yearly = wmt#.groupby("time.year").mean("time").compute()
        
        decompose_boundary_fluxes(wmt_yearly)
        
        wmb_decomp = xwmb.WaterMassBudget(
            grid,
            budgets_dict, 
            region = region, 
            decompose=["surface_exchange_flux", "nonadvective", "advective"]
        ) #if region not passed, the whole globe is taken
        
        wmb_decomp.mass_budget("sigma2", greater_than=True, default_bins=False, 
                        integrate=False, along_section=False)
        
        wmt_decomp = wmb_decomp.wmt[decomp_budget_terms].sel(selection_kwargs).rename({"sigma2_l_target":"sigma2_l"})
        wmt_decomp_yearly = wmt_decomp#.groupby("time.year").mean("time").compute()
        
        wmts += [1 * xr.merge([wmt_decomp_yearly, wmt_yearly, ds["thkcello"].sel(ds_kwargs)])]



/proj/ecco/CM4X/budget_sigma2_v1.2.0/CM4Xp125_budgets_sigma2_1850-1854.zarr
Inferring Z grid coordinate: density `sigma2`
/proj/ecco/CM4X/budget_sigma2_v1.2.0/CM4Xp125_budgets_sigma2_1855-1859.zarr
Inferring Z grid coordinate: density `sigma2`


In [46]:
import sys
sys.path.insert(0, '/vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/src')
from src import *
from SWMT_decomposition import *

concat_wmt = xr.concat(wmts, dim = "time")
ds_SWMT = get_SWMT(concat_wmt)
ds_SWMT["thkcello"] = concat_wmt["thkcello"]

In [47]:
SWMT = ds_SWMT[["SWMT", "thkcello"]].compute()

In [67]:
sel_SWMT = SWMT.sel(sigma2_l = 37.00, method = "nearest").isel(exp = 0).isel(time = [0, 1]).fillna(0.0)

net_SWMT = sel_SWMT["SWMT"].sum(["xh", "yh"]).compute()

SWMT_diff = net_SWMT.isel(time = -1) - net_SWMT.isel(time = 0)

print(SWMT_diff.compute().values)


eps = 1e-12

S = sel_SWMT["SWMT"]
h = sel_SWMT["thkcello"]

G = xr.where(h > eps, S / h, 0.0)


dG = G.isel(time = -1) - G.isel(time = 0)

dh = h.isel(time = -1) - h.isel(time = 0)

h0 = h.isel(time = 0)
G0 = G.isel(time = 0)

dg_contrib = (dG * h0).sum(["xh", "yh"]).compute()
dh_contrib = (G0 * dh).sum(["xh", "yh"]).compute()

dgdh_contrib = (dG * dh).sum(["xh", "yh"]).compute()

print(dg_contrib.values)
print(dh_contrib.values)
print(dgdh_contrib.values)

print((dg_contrib + dh_contrib + dgdh_contrib).values)


1278982294.1668482
1028051032.0614004
-387415081.82000697
638346343.9254546
1278982294.1668482


In [70]:
dims = ["xh", "yh"]
eps = 1e-12

sel = (
    SWMT
    .sel(sigma2_l=37.00, method="nearest")
    .isel(exp=0)
)

S = sel["SWMT"].fillna(0.0)
h = sel["thkcello"].fillna(0.0)

G = xr.where(h > eps, S / h, 0.0)

Gbar = G.groupby("time.year").mean("time")
hbar = h.groupby("time.year").mean("time")

G_anom = G.groupby("time.year") - Gbar
h_anom = h.groupby("time.year") - hbar

cov = (G_anom * h_anom).groupby("time.year").mean("time")

Ghbar = (G * h).groupby("time.year").mean("time")
cov_check = Ghbar - Gbar * hbar

print((cov - cov_check).max().compute().values)
print((cov - cov_check).min().compute().values)

G0 = Gbar.isel(year=0)
G1 = Gbar.isel(year=-1)

h0 = hbar.isel(year=0)
h1 = hbar.isel(year=-1)

cov0 = cov.isel(year=0)
cov1 = cov.isel(year=-1)

dG = G1 - G0
dh = h1 - h0

dG_contrib = (h0 * dG).sum(dims)
dh_contrib = (G0 * dh).sum(dims)
dGdh_contrib = (dG * dh).sum(dims)
cov_contrib = (cov1 - cov0).sum(dims)

closure = dG_contrib + dh_contrib + dGdh_contrib + cov_contrib

SWMT_y = S.groupby("time.year").mean("time").sum(dims)
SWMT_diff = SWMT_y.isel(year=-1) - SWMT_y.isel(year=0)

def sci(x, precision=6):
    return f"{float(x.compute().values):.{precision}e}"

print("SWMT_diff    =", sci(SWMT_diff))
print("dG contrib   =", sci(dG_contrib))
print("dh contrib   =", sci(dh_contrib))
print("dGdh contrib =", sci(dGdh_contrib))
print("cov contrib  =", sci(cov_contrib))
print("closure      =", sci(closure))
print("residual     =", sci(SWMT_diff - closure))

1.1920928955078125e-07
-1.1920928955078125e-07
SWMT_diff    = -2.869962e+09
dG contrib   = -1.753482e+09
dh contrib   = -1.202118e+09
dGdh contrib = 3.280132e+08
cov contrib  = -2.423764e+08
closure      = -2.869962e+09
residual     = 9.536743e-07


In [ ]:
wmts_ds = xr.concat(wmts, dim = "year")
print(f"Dataset size: {wmts_ds.nbytes / (1024**3):.2f} GB")
savedir = "/vortexfs1/home/anthony.meza/scratch/CM4XAbyssalSWMT/data/model/"
wmts_ds.to_zarr(savedir + f"Southern_Ocean_DSW_1000m_WMT_Surface_Fluxes_sigma_slice.zarr", mode = "w")

In [ ]:
sigma = 36.974

fig, ax = plt.subplots(1, 2, figsize = (12, 5))
wmts_ds.surface_exchange_flux_salt.isel(exp = 0).mean("year").sel(
                                                                  sigma2_l_target = sigma, method = "nearest").plot(x = "geolon", y = "geolat", ax = ax[0])
wmts_ds.surface_exchange_flux_heat.isel(exp = 0).mean("year").sel(
                                                                  sigma2_l_target = sigma, method = "nearest").plot(x = "geolon", y = "geolat", ax = ax[1])

In [ ]:
sigma = 36.974

fig, ax = plt.subplots(1, 2, figsize = (12, 5))
wmts_ds.surface_exchange_flux_salt.isel(exp = 0).mean("year").sel(
                                                                  sigma2_l_target = sigma, method = "nearest").plot(x = "geolon", y = "geolat", ax = ax[0])
wmts_ds.surface_exchange_flux_heat.isel(exp = 0).mean("year").sel(
                                                                  sigma2_l_target = sigma, method = "nearest").plot(x = "geolon", y = "geolat", ax = ax[1])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (12, 5))
wmts_ds.isel(exp = 1).mean("year").surface_exchange_flux_salt.sel(sigma2_l_target = sigma, method = "nearest").plot(x = "geolon", y = "geolat", ax = ax[0])
wmts_ds.isel(exp = 1).mean("year").surface_exchange_flux_heat.sel(sigma2_l_target = sigma, method = "nearest").plot(x = "geolon", y = "geolat", ax = ax[1])